# RSNA Knee — Final Evidence-First Multi-Family Ensemble

This notebook is the final revision of `rsna-knee-enhanced-ensemble`.

It keeps the checkpoint-compatible DINOv2 preprocessing/model contract intact, but replaces
the unvalidated scalar-quality hybrid as the primary path with the strongest reproducible
lessons found in the supplied notebooks:

1. **Diagnosis-specific window pooling** for focal findings (`max` / `top2`) before
   cross-member rank averaging.
2. **Equal member rank voting** as the DINO safety ensemble, avoiding unsupported
   cross-fold score weighting when per-target OOF metadata is absent.
3. **Optional independent EfficientNet-B3 diversity blend** when a complete five-fold
   B3 package is attached and its audit supports the blend.
4. Every aggressive candidate is written separately; `submission.csv` is promoted only
   through an evidence/audit gate.

The design is intended to improve the 0.891 DINO inference path. A 0.95 leaderboard score
cannot be guaranteed without validating on the competition leaderboard and, more
importantly, without stronger independently trained model families / supervision.


## Required Kaggle inputs

Required:

1. **RSNA Knee competition data**
   - `test.csv`
   - `test_series.csv`
   - `test_series/.../*.dcm`

2. **20-member manifest DINOv2 weight package**
   - `manifest.json`
   - every checkpoint referenced by `manifest["members"]`

3. **Offline DINOv2-small model directory**
   - Hugging Face-style directory containing `config.json`

Optional but recommended for the multi-family candidate:

4. **Five-fold EfficientNet-B3 package** compatible with the supplied V47/V49 recipe
   - default search path: `/kaggle/input/rsna-knee-b3-v47-folds-0-3`
   - `source/efficientnet_b3_public_repro_v4_t4.py`
   - `source/efficientnet_b3_public_repro_v1_infer.py`
   - `fold0/fold0_final.pt` ... `fold4/fold4_final.pt`
   - preferably `audit/audit.json`

Environment overrides:

- `KNEE_INPUT_DIR`
- `KNEE_WEIGHTS_DIR`
- `KNEE_DINOV2_DIR`
- `KNEE_B3_DIR`
- `RSNA_FINAL_MODE=auto|dino_frontier|dino_b3_10`
- `ALLOW_UNAUDITED_B3=0|1` (default `0`)

`auto` always produces the DINO frontier first. It promotes the 10% B3 rank blend only if
the B3 package completes successfully and its audit shows nested OOF improvement over the
DINO reference (or the user explicitly opts into an unaudited blend).


## 1. Configuration — keep the original dataset/model mounts

In [ ]:
from __future__ import annotations

import os
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(_v, "4")

import gc
import hashlib
import json
import re
import time
import traceback
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd
import pydicom
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import display

T0 = time.time()
SEED = 2026
np.random.seed(SEED)
torch.manual_seed(SEED)

TARGETS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA",
    "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's",
    "Contusion", "Fracture",
]

# -----------------------------------------------------------------------------
# IMPORTANT: these are the same competition/model conventions used by the
# 0.891 public pipeline. The manifest will overwrite pixel settings member by
# member before decoding, so do not "improve" crop/order/laterality independently
# of the checkpoint that was trained on them.
# -----------------------------------------------------------------------------
_weights_override = os.environ.get("KNEE_WEIGHTS_DIR", "").strip()
PREFERRED_WEIGHTS = Path(_weights_override) if _weights_override else Path("/kaggle/input/rsna-knee-weights")
CROP_MM = 130.0
CACHE_IMG = 336
IMG = CACHE_IMG
GROUP = 3
CACHE_SLICES = 12
N_GROUP = max(CACHE_SLICES // GROUP, 1)
SLICE_BAND = (0.20, 0.80)

HDR_THREADS = 16
PIX_THREADS = 12
ORDER_THREADS = 32
ORDER_BUDGET_S = 5400
EVAL_BATCH = 8
TIME_BUDGET = 8.0 * 3600
LAT_MIN_OFFSET_MM = 20.0
LEGACY_LAT_OFFSET_MM = 5.0

RULES_NATIVE = {
    "order": "normal",
    "lat": "centre",
    "slot_fallback": False,
    "decode_fill": "nearest",
}
RULES_LEGACY = {
    "order": "dominant_axis",
    "lat": "corner_x",
    "slot_fallback": True,
    "decode_fill": "zero",
}
RULES = dict(RULES_NATIVE)

SLOTS_RECOVERED = [
    ("SAG_FLUID_FS", "Sagittal", True, True),
    ("COR_FLUID_FS", "Coronal", True, True),
    ("AX_FLUID_FS", "Axial", True, True),
    ("SAG_FLUID_NOFS", "Sagittal", True, False),
    ("COR_T1", "Coronal", False, False),
    ("SAG_T1", "Sagittal", False, False),
]
SLOTS_PUBLIC = [
    ("SAG_FLUID", "Sagittal", None, True),
    ("COR_FLUID", "Coronal", None, True),
    ("AX_FLUID", "Axial", None, True),
    ("SAG_STRUCT", "Sagittal", None, False),
    ("COR_STRUCT", "Coronal", None, False),
    ("AX_STRUCT", "Axial", None, False),
]
SLOT_SCHEME = os.environ.get("SLOT_SCHEME", "recovered")
SLOTS = SLOTS_PUBLIC if SLOT_SCHEME == "public" else SLOTS_RECOVERED
N_SLOT = len(SLOTS)

POOL_PARTS = {"cls_mean": 2, "cls_mean_focal": 3}
SLOT_PRIOR_TABLE = {
    "ACL": (0, 3, 5), "MCL": (1, 4),
    "Medial Meniscus": (0, 1, 3, 4), "Lateral Meniscus": (0, 1, 3, 4),
    "Medial OA": (1, 4, 5), "Lateral OA": (1, 4, 5),
    "PF OA": (0, 2, 5), "Effusion": (0, 2), "Synovitis": (0, 2),
    "Baker's": (0,), "Contusion": (0, 1, 2), "Fracture": (0, 1, 2, 4, 5),
}
SLOT_PRIOR_STRENGTH = 0.55

FATSAT_OPTS = {"FS", "FATSAT", "FAT_SAT", "FSAT"}
_SEP = re.compile(r"[_\-.]")
_FATSAT_RX = re.compile(
    r"\bfs\b|fatsat|fat sat|\bstir\b|\bspair\b|\bspir\b|\bwe\b|"
    r"water excit|\btirm\b|\bsting\b|\bfatsup\b"
)
_T1_RX = re.compile(r"\bt1\b|\bt1w\b")
_T2_RX = re.compile(r"\bt2\b|\bt2w\b")
_PD_RX = re.compile(r"\bpd\b|\bpdw\b|proton|\bdp\b|dens")

# Evidence-first final ensemble controls.
# The 20-member DINO frontier is the mandatory safety arm. The independent B3 family is
# promoted only through an audit gate.
FINAL_MODE = os.environ.get("RSNA_FINAL_MODE", "auto").lower()
B3_GLOBAL_ALPHA = float(os.environ.get("RSNA_B3_ALPHA", "0.10"))
ALLOW_UNAUDITED_B3 = os.environ.get("ALLOW_UNAUDITED_B3", "0").strip() == "1"

# Diagnosis-specific pooling measured in the supplied higher-scoring DINO notebook.
# Focal findings benefit from retaining the strongest local window rather than diluting
# it across the whole stack; ACL/MCL use top-2 evidence for more stability.
FRONTIER_TARGET_POOL = {
    "Fracture": "max",
    "Contusion": "max",
    "Medial Meniscus": "max",
    "Lateral Meniscus": "max",
    "ACL": "top2",
    "MCL": "top2",
    "Baker's": "max",
}

# Target-wise B3 weights are kept only as a secondary candidate because they were selected
# on a very small expert-labelled set. The safer primary cross-family candidate is 10%.
B3_TARGET_ALPHAS = {
    "ACL": 0.00,
    "MCL": 0.10,
    "Medial Meniscus": 0.00,
    "Lateral Meniscus": 0.35,
    "Medial OA": 0.15,
    "Lateral OA": 0.35,
    "PF OA": 0.35,
    "Effusion": 0.25,
    "Synovitis": 0.35,
    "Baker's": 0.35,
    "Contusion": 0.00,
    "Fracture": 0.00,
}


In [ ]:
def log(msg):
    print(f"[{time.time() - T0:7.1f}s] {msg}", flush=True)


def find_root():
    for c in [
        Path("/kaggle/input/competitions/rsna-knee-abnormality-detection"),
        Path("/kaggle/input/rsna-knee-abnormality-detection"),
        Path("data"),
        Path("."),
    ]:
        if (c / "test.csv").is_file() and (c / "test_series").is_dir():
            return c
    base = Path("/kaggle/input")
    if base.is_dir():
        for depth1 in sorted(p for p in base.iterdir() if p.is_dir()):
            for cand in [depth1] + sorted(p for p in depth1.iterdir() if p.is_dir()):
                if (cand / "test.csv").is_file() and (cand / "test_series").is_dir():
                    return cand
    raise FileNotFoundError("RSNA knee competition mount not found")



def _looks_like_dinov2_config(config_path: Path):
    try:
        cfg = json.loads(config_path.read_text())
    except Exception:
        return False
    text = json.dumps(cfg).lower()
    return (
        cfg.get("model_type", "").lower() == "dinov2"
        or "dinov2" in text
        or any("dinov2" in str(x).lower() for x in cfg.get("architectures", []) or [])
    )


def find_dinov2(variant="small"):
    """Find a local/offline Hugging Face DINOv2 model directory.

    Supports an explicit KNEE_DINOV2_DIR override and then scans mounted Kaggle
    inputs. It does not require the folder name itself to contain 'dinov2'.
    """
    explicit = os.environ.get("KNEE_DINOV2_DIR", "").strip()
    if explicit:
        p = Path(explicit)
        if (p / "config.json").is_file():
            return p
        raise FileNotFoundError(
            f"KNEE_DINOV2_DIR={explicit!r} does not contain config.json"
        )

    base = Path("/kaggle/input")
    if not base.is_dir():
        return None

    strong, weak = [], []
    for root, dirs, files in os.walk(base):
        dirs[:] = [d for d in dirs if d not in ("train_series", "test_series")]
        if "config.json" not in files:
            continue
        p = Path(root)
        cfg = p / "config.json"
        if _looks_like_dinov2_config(cfg):
            strong.append(p)
        elif "dinov2" in str(p).lower():
            weak.append(p)

    hits = strong or weak
    if not hits:
        return None

    variant_l = str(variant).lower()
    for p in hits:
        if variant_l in str(p).lower():
            return p

    # Match common DINOv2 hidden sizes when folder naming is generic.
    target_hidden = {"small": 384, "base": 768, "large": 1024, "giant": 1536}.get(variant_l)
    if target_hidden is not None:
        for p in hits:
            try:
                cfg = json.loads((p / "config.json").read_text())
                if int(cfg.get("hidden_size", -1)) == target_hidden:
                    return p
            except Exception:
                pass

    return hits[0]


ROOT = find_root()
log(f"input root: {ROOT}")
_dino_probe = find_dinov2("small")
log(f"DINOv2 probe: {_dino_probe}")


## 2. Lightweight EDA and annotation sanity checks

In [ ]:
def read_csv_if_present(name):
    p = ROOT / name
    if not p.is_file():
        return pd.DataFrame()
    return pd.read_csv(p)


train_df_eda = read_csv_if_present("train.csv")
train_series_eda = read_csv_if_present("train_series.csv")
test_df_eda = read_csv_if_present("test.csv")
test_series_eda = read_csv_if_present("test_series.csv")

summary = []
for name, frame in [
    ("train.csv", train_df_eda),
    ("train_series.csv", train_series_eda),
    ("test.csv", test_df_eda),
    ("test_series.csv", test_series_eda),
]:
    if not frame.empty:
        summary.append({"file": name, "rows": len(frame), "columns": len(frame.columns)})
display(pd.DataFrame(summary))

# Correct prevalence: missing labels are unknown, NOT negative.
present_targets = [t for t in TARGETS if t in train_df_eda.columns]
if present_targets:
    n_labeled = train_df_eda[present_targets].notna().sum()
    positives = train_df_eda[present_targets].sum(skipna=True)
    negatives = n_labeled - positives
    stats = pd.DataFrame({
        "annotated": n_labeled,
        "positive": positives,
        "negative": negatives,
        "positive_rate_on_annotated": positives / n_labeled.replace(0, np.nan),
    })
    display(stats.style.format({"positive_rate_on_annotated": "{:.1%}"}))
    fully = int(train_df_eda[present_targets].notna().all(axis=1).sum())
    print(f"Studies with all 12 image annotations: {fully} / {len(train_df_eda)}")

if not train_series_eda.empty:
    cols = [c for c in ["Fluid_Sensitive", "Fat_Suppression", "Anatomical_Plane"]
            if c in train_series_eda.columns]
    if cols:
        display(train_series_eda.groupby(cols, dropna=False).size()
                .rename("series_count").reset_index()
                .sort_values("series_count", ascending=False).head(30))
    per_study = train_series_eda.groupby("StudyInstanceUID").size()
    display(per_study.describe().rename("series_per_study").to_frame())



**Important:** the competition CSV sequence flags are audited, not forced into the imported 0.891 checkpoints. Their manifest records the exact recovered sequence/preprocessing contract used during training; changing that only at inference would be a train/test preprocessing mismatch.


## 3. DICOM acquisition metadata and recovered sequence semantics

In [ ]:
HDR_TAGS = ["SeriesDescription", "SequenceName", "ScanOptions", "ScanningSequence",
            "RepetitionTime", "EchoTime", "Laterality", "PixelSpacing", "Rows",
            "Columns", "RescaleSlope", "RescaleIntercept",
            # Position and orientation are read from the same header probe() already
            # opens, so they cost nothing, and they are what recovers the side when the
            # Laterality tag is absent - which it is for half the studies here.
            "ImagePositionPatient", "ImageOrientationPatient"]


def _hdr_vec(s, n):
    """Parse a DICOM multi-value string as stored by probe(): floats joined by `|`."""
    if not isinstance(s, str):
        return None
    try:
        v = [float(x) for x in s.split("|")]
    except ValueError:
        return None
    return np.array(v) if len(v) >= n else None


def side_from_geometry(h):
    """Study -> 'L' / 'R' / None, from where the image sits in the patient.

    `Laterality` (0020,0060) is Type 2C and may legitimately be absent; in this corpus it
    is missing on exactly half the studies, and the vendors it is missing from are whole
    vendors rather than scattered series. A study with no tag is not a left knee, but the
    normalisation upstream treats it as one, so half the corpus was never normalised and
    the five side-defined targets - the two menisci, the two tibiofemoral compartments
    and the medial collateral ligament - saw that axis reversed on a large minority of it.

    The patient coordinate system fixes this without the tag: +x is the patient's left, so
    the centre of a right knee sits at negative x. The centre is used rather than
    `ImagePositionPatient` itself because that is the corner of the image, which is offset
    by half a field of view - enough to change the sign on a knee near the midline.

    The median over a study's series is what is thresholded, not a single series: probe()
    reads one arbitrary slice per series, which on a sagittal stack can sit anywhere
    across the joint. Studies whose centre falls near the midline are left unresolved
    rather than guessed - measured against the tagged half, the rule is right 97% of the
    time overall and no better than chance inside 20 mm.
    """
    cx = {}
    for r in h.itertuples(index=False):
        ipp = _hdr_vec(getattr(r, "ImagePositionPatient", None), 3)
        iop = _hdr_vec(getattr(r, "ImageOrientationPatient", None), 6)
        ps = _hdr_vec(getattr(r, "PixelSpacing", None), 2)
        rows, cols = getattr(r, "Rows", None), getattr(r, "Columns", None)
        if ipp is None or iop is None or ps is None or not rows or not cols:
            continue
        try:
            c = ipp[:3] + iop[:3] * ps[1] * float(cols) / 2 + iop[3:6] * ps[0] * float(rows) / 2
        except (TypeError, ValueError):
            continue
        cx.setdefault(r.StudyInstanceUID, []).append(float(c[0]))
    out = {}
    for st, xs in cx.items():
        m = float(np.median(xs))
        out[st] = None if abs(m) < LAT_MIN_OFFSET_MM else ("R" if m < 0 else "L")
    return out


def side_from_corner_x(h):
    """The laterality an imported member was fitted under.

    It thresholds the median raw `ImagePositionPatient` x over a study's series. That is
    the x of the image *corner*, not of its centre, so it differs from the rule above by
    up to half a field of view - which is enough to reverse the sign on a knee scanned
    near the midline. The dead zone is 5 mm rather than 20 mm, so it also commits on
    studies the rule above leaves unresolved.

    Neither difference changes a shape. Each one decides whether a study is mirrored, and
    a study mirrored one way at training and the other at inference presents the five
    side-defined targets with their axis reversed.
    """
    out = {}
    for st, g in h.groupby("StudyInstanceUID"):
        xs = []
        for r in g.itertuples(index=False):
            ipp = _hdr_vec(getattr(r, "ImagePositionPatient", None), 3)
            if ipp is not None and np.isfinite(ipp).all():
                xs.append(float(ipp[0]))
        if not xs:
            out[st] = None
            continue
        x = float(np.median(xs))
        # DICOM patient coordinates are LPS: +x is the patient's left.
        out[st] = None if abs(x) < LEGACY_LAT_OFFSET_MM else ("R" if x < 0 else "L")
    return out


def lat_of(h, tag=""):
    """Study -> 'L' / 'R' / None: the tag where it exists, geometry where it does not.

    The tag is present on exactly half the studies here and is sometimes an empty
    string rather than absent, which is not the same as NaN. Treating the other half
    as left-sided is what `normalise_laterality` did by omission, so the geometry
    fallback is not a refinement - it is the difference between normalising half the
    corpus and normalising all of it.
    """
    geo = side_from_corner_x(h) if RULES["lat"] == "corner_x" else side_from_geometry(h)
    d, n_tag, n_geo, n_none, n_disagree = {}, 0, 0, 0, 0
    for st, g in h.groupby("StudyInstanceUID"):
        v = [str(x).strip().upper() for x in g["Laterality"].dropna()]
        if RULES["lat"] == "corner_x" and "ImageLaterality" in g.columns:
            # The legacy rule reads the second tag too, so a study tagged only there is
            # resolved from the tag rather than from geometry.
            v += [str(x).strip().upper() for x in g["ImageLaterality"].dropna()]
        v = [x[0] for x in v if x and x[0] in ("L", "R")]
        side = v[0] if v else None
        if side is not None:
            n_tag += 1
            if geo.get(st) is not None and geo[st] != side:
                n_disagree += 1
        else:
            side = geo.get(st)
            n_geo += side is not None
            n_none += side is None
        d[st] = side
    log(f"{tag}laterality: {n_tag} from the tag, {n_geo} from geometry, "
        f"{n_none} unresolved; tag and geometry disagree on {n_disagree} "
        f"({n_disagree / max(n_tag, 1):.1%} of the tagged)")
    return d



def probe(item):
    split, study, series, path = item
    row = {"split": split, "StudyInstanceUID": study, "SeriesInstanceUID": series,
           "dir": path}
    try:
        files = sorted(e.name for e in os.scandir(path) if e.name.endswith(".dcm"))
        row["files"] = files
        row["n_slices"] = len(files)
        if not files:
            return row
        ds = pydicom.dcmread(os.path.join(path, files[len(files) // 2]),
                             stop_before_pixels=True, force=True)
        for t in HDR_TAGS:
            v = getattr(ds, t, None)
            if v is None:
                row[t] = None
            elif isinstance(v, (list, tuple)) or type(v).__name__ == "MultiValue":
                row[t] = "|".join(str(x) for x in v)
            else:
                row[t] = str(v)
    except Exception as exc:
        row["err"] = str(exc)[:120]
    return row


def walk(split):
    """Every series directory of a split, with one header read per series.

    An absent split returns an empty frame *with the columns annotate expects*. Returning
    a bare DataFrame looks like the same thing and is not: the next call indexes
    `SeriesDescription` and raises KeyError, so the branch that exists to survive a
    missing split is what turns it into a crash.
    """
    base = ROOT / split
    items = []
    if not base.is_dir():
        return pd.DataFrame(columns=["split", "StudyInstanceUID", "SeriesInstanceUID",
                                     "dir", "files", "n_slices"] + HDR_TAGS)
    for study in os.scandir(base):
        if study.is_dir():
            for series in os.scandir(study.path):
                if series.is_dir():
                    items.append((split, study.name, series.name, series.path))
    with ThreadPoolExecutor(max_workers=HDR_THREADS) as pool:
        rows = list(pool.map(probe, items))
    return pd.DataFrame(rows)


def annotate(df):
    """Recover fat suppression and pulse-sequence weighting from the header."""
    desc = (df["SeriesDescription"].fillna("") + " " + df["SequenceName"].fillna(""))
    desc = desc.str.lower().str.replace(_SEP, " ", regex=True)

    opts = df["ScanOptions"].fillna("").str.upper().str.split("|")
    # GE writes SAT_GEMS for spatial saturation, so ScanOptions must be matched as
    # exact tokens; a substring test on "SAT" fires on non-fat-sat series.
    opts_fs = opts.apply(lambda ts: any(t.strip() in FATSAT_OPTS for t in ts))
    df["fatsat"] = desc.str.contains(_FATSAT_RX) | opts_fs

    tr = pd.to_numeric(df["RepetitionTime"], errors="coerce")
    te = pd.to_numeric(df["EchoTime"], errors="coerce")
    gre = df["ScanningSequence"].fillna("").str.upper().str.contains("GR")
    t1, t2, pdw = desc.str.contains(_T1_RX), desc.str.contains(_T2_RX), desc.str.contains(_PD_RX)

    df["weight"] = np.where(t1 & ~t2 & ~pdw, "T1",
                     np.where(t2 & ~pdw, "T2",
                       np.where(pdw, "PD",
                         np.where(gre, "GRE",
                           np.where(tr < 800, "T1",
                             np.where(te > 60, "T2",
                               np.where(tr >= 800, "PD", "UNK")))))))
    df["fluid"] = np.isin(df["weight"], ["PD", "T2"])
    df["px"] = pd.to_numeric(
        df["PixelSpacing"].fillna("").str.split("|").str[0].replace("", np.nan),
        errors="coerce")
    return df


In [ ]:
def _as_bool(v):
    if pd.isna(v):
        return None
    s = str(v).strip().upper()
    if s in {"1", "TRUE", "T", "YES", "Y"}:
        return True
    if s in {"0", "FALSE", "F", "NO", "N"}:
        return False
    try:
        f = float(s)
        return True if f == 1 else False if f == 0 else None
    except Exception:
        return None


def audit_official_sequence_metadata(inferred, official):
    """Audit only. Do not change imported checkpoint pixels from this table."""
    need = {"SeriesInstanceUID", "Fluid_Sensitive", "Fat_Suppression"}
    if inferred.empty or official.empty or not need.issubset(official.columns):
        return
    a = inferred[["SeriesInstanceUID", "fluid", "fatsat"]].copy()
    b = official[["SeriesInstanceUID", "Fluid_Sensitive", "Fat_Suppression"]].copy()
    b["official_fluid"] = b["Fluid_Sensitive"].map(_as_bool)
    b["official_fatsat"] = b["Fat_Suppression"].map(_as_bool)
    m = a.merge(b[["SeriesInstanceUID", "official_fluid", "official_fatsat"]],
                on="SeriesInstanceUID", how="inner")
    for inferred_col, official_col, name in [
        ("fluid", "official_fluid", "Fluid_Sensitive"),
        ("fatsat", "official_fatsat", "Fat_Suppression"),
    ]:
        valid = m[official_col].notna() & m[inferred_col].notna()
        if valid.any():
            agree = (m.loc[valid, inferred_col].astype(bool).values ==
                     m.loc[valid, official_col].astype(bool).values).mean()
            log(f"metadata audit {name}: {agree:.1%} agreement on {int(valid.sum())} series")


## 4. Select the six MRI slots exactly as the checkpoint expects

In [ ]:
def pick_slots(series_df, plane_map):
    """One series per slot per study.

    Ties are broken toward the stack with the most slices: a thicker stack samples the
    joint more densely, and the three-slice sampler below benefits from the margin.
    """
    series_df = series_df.copy()
    series_df["plane"] = series_df["SeriesInstanceUID"].map(plane_map)
    out = {}
    for study, g in series_df.groupby("StudyInstanceUID"):
        chosen = {}
        for name, plane, fluid, fs in SLOTS:
            sel = (g["plane"] == plane) & (g["fatsat"] == fs)
            # fluid=None means "do not condition on weighting" - the public scheme,
            # where the single provided flag stands in for both axes at once.
            if fluid is not None:
                sel &= (g["fluid"] == fluid)
            cand = g[sel]
            # A slot with no series matching its predicate stays empty, and no substitute
            # is admitted from a neighbouring predicate. Relaxing the weighting to fill a
            # T1 slot would draw from the pool `SAG_FLUID_NOFS` selects from, since that
            # pool is what remains once the weighting is dropped: over the training corpus
            # it would put one series in two slots for 2383 of 4407 studies and leave 56%
            # of the T1 slot holding PD or T2. The presence mask would then assert a
            # sequence that was never acquired, and the per-diagnosis softmax of §6 would
            # divide its attention across two identical slots, giving one acquisition
            # about twice the weight it carries in a study that holds both. The mask is
            # there to say a slot is absent, which is what an absent slot is.
            if len(cand) == 0 and RULES["slot_fallback"] and fluid is False:
                # The relaxation the paragraph above rejects, reproduced because an
                # imported member was fitted with its T1 slots filled this way: over half
                # of that member's training studies had a T1 slot holding a series that
                # is not T1. Leaving those slots empty would present it with a presence
                # mask it never saw.
                cand = g[(g["plane"] == plane) & (~g["fatsat"])]
            if len(cand):
                chosen[name] = cand.sort_values("n_slices", ascending=False).iloc[0]
        out[study] = chosen
    return out


## 5. Physical slice ordering, crop, intensity normalisation

In [ ]:
ORDER_TAGS = [(0x0020, 0x0032), (0x0020, 0x0037), (0x0020, 0x0013)]

# Series in which at least one sampled slice would not decode. A list rather than a
# counter because appending is atomic under the reader threads, and reported rather than
# swallowed: unreported, a decode failure is indistinguishable from a black knee.
DECODE_FAILED = []


def cache_tag(rules=None):
    """The name a decoded cache is stored under.

    It has to name everything that decides the pixels, not only their dimensions. Two
    configurations that agree on resolution, slice count, crop and band but disagree on
    how a slice is chosen produce different arrays of identical shape - so a tag built
    from the dimensions alone lets the second attach to the first one's file and train
    against pixels it never asked for, with nothing anywhere reporting a mismatch.

    A native reading keeps the plain name, so caches decoded before the rules existed
    stay valid; anything else earns a suffix.
    """
    r = dict(RULES if rules is None else rules)
    t = (f"{CACHE_IMG}px_{CACHE_SLICES}sl_{int(CROP_MM)}mm_"
         f"{SLICE_BAND[0]:.2f}-{SLICE_BAND[1]:.2f}")
    if {k: r.get(k, v) for k, v in RULES_NATIVE.items()} != RULES_NATIVE:
        t += "_" + hashlib.md5(json.dumps(r, sort_keys=True).encode()).hexdigest()[:6]
    return t


def _natural_key(name):
    return tuple(int(x) if x.isdigit() else x.lower()
                 for x in re.split(r"(\d+)", str(name)))


def _order_dominant_axis(rec):
    """The slice order an imported member was fitted under.

    It sorts on the raw patient coordinate along whichever axis varies most across the
    stack, rather than on the projection onto the slice normal. The two differ by a sign,
    not by a formula: measured over this corpus every sagittal series has a slice normal
    with n_x in [-1.00, -0.98], so p.n is the negative of the raw x this sorts on and the
    two stacks come out exactly reversed. Because the band sampler truncates rather than
    rounds, its nine indices are not symmetric about the middle, so nine slices drawn from
    a twenty-six slice stack under one order share two with the other.

    Missing geometry falls back to `InstanceNumber` and then to a natural sort of the file
    name, both at the same 80% threshold the imported pipeline used.
    """
    files, d = rec["files"], rec["dir"]
    rows = []
    for pos, f in enumerate(files):
        ipp = inst = None
        try:
            ds = pydicom.dcmread(os.path.join(d, f), force=True, stop_before_pixels=True,
                                 specific_tags=["ImagePositionPatient", "InstanceNumber"])
            raw = getattr(ds, "ImagePositionPatient", None)
            if raw is not None and len(raw) >= 3:
                c = np.asarray(raw[:3], dtype=np.float64)
                if np.isfinite(c).all():
                    ipp = c
            n = getattr(ds, "InstanceNumber", None)
            if n is not None:
                inst = float(n)
        except Exception:
            pass
        rows.append((f, ipp, inst, pos))

    placed = [r for r in rows if r[1] is not None]
    need = max(2, int(0.8 * len(rows)))
    if len(placed) >= need:
        xyz = np.stack([r[1] for r in placed])
        axis = int(np.argmax(np.ptp(xyz, axis=0)))
        spare = float(np.nanmedian(xyz[:, axis]))
        rows.sort(key=lambda r: (float(r[1][axis]) if r[1] is not None else spare,
                                 r[2] if r[2] is not None else float("inf"), r[3]))
    elif sum(r[2] is not None for r in rows) >= need:
        rows.sort(key=lambda r: (r[2] if r[2] is not None else float("inf"), r[3]))
    else:
        rows.sort(key=lambda r: _natural_key(r[0]))
    return [r[0] for r in rows], True


def order_slices(rec):
    """Return the series' files sorted along the through-plane axis.

    A DICOM file name here is a SOP Instance UID, which is assigned arbitrarily. Sorting
    by it therefore produces an order uncorrelated with anatomy - measured over one
    series, Spearman between file-name rank and physical position is 0.009, i.e. none.
    Anything that assumes the file order means something is then operating on noise: the
    three channels of a "2.5D" input are three unrelated views rather than neighbouring
    slices, "the middle of the stack" is a random subset, and reversing slice order to
    normalise laterality reverses nothing meaningful.

    The physical order is recoverable exactly. Each slice carries its position in patient
    coordinates and the in-plane axes; projecting the position onto the slice normal
    gives a signed through-plane coordinate, monotonic along the stack:

        n = r_x  x  r_y ,      k = p . n

    `InstanceNumber` is the fallback. It usually tracks the projection up to sign, but
    interleaved and multi-echo acquisitions need not number slices in the order they
    occupy in space - but the projection is signed in patient
    coordinates, which is what laterality normalisation needs.
    """
    if RULES["order"] == "dominant_axis":
        return _order_dominant_axis(rec)
    files, d = rec["files"], rec["dir"]
    keyed = []
    for f in files:
        k = None
        try:
            ds = pydicom.dcmread(os.path.join(d, f), force=True, stop_before_pixels=True,
                                 specific_tags=ORDER_TAGS)
            iop = np.asarray(ds.ImageOrientationPatient, dtype=float)
            ipp = np.asarray(ds.ImagePositionPatient, dtype=float)
            k = float(np.dot(ipp, np.cross(iop[:3], iop[3:])))
        except Exception:
            try:
                k = float(ds.InstanceNumber)
            except Exception:
                k = None
        keyed.append((k, f))
    if any(k is None for k, _ in keyed):
        # A series with no usable geometry keeps its arbitrary order; that is worse than
        # sorting but better than dropping the series, and it is logged as a count.
        return files, False
    return [f for _, f in sorted(keyed, key=lambda t: t[0])], True


def read_slot(rec, n_slice=None, out_size=None):
    """`n_slice` physically spread slices from one series, at `out_size` pixels.

    Returns uint8 [n_slice, out, out] normalised per-series to its 1st-99th
    percentile. Percentiles rather than min/max because MR intensity has no absolute
    scale and a single bright vessel would otherwise compress the whole dynamic range.

    Reading is the expensive half of this pipeline, so the caller reads once at the
    largest configuration it needs and derives the smaller ones from the returned buffer
    rather than re-reading.
    """
    n_slice = GROUP if n_slice is None else n_slice
    out_size = IMG if out_size is None else out_size
    files, d, px = rec.get("ordered") or rec["files"], rec["dir"], rec["px"]
    n = len(files)
    if n == 0:
        return None
    # Spread the samples over a central band of the stack: the outermost slices of a knee
    # series are mostly soft tissue outside the joint. The band is a constant rather than
    # a literal because how much of the stack is worth reading depends on how many slices
    # are being taken - at three the middle is all that fits, while at sixteen the ends
    # are worth having, and a Baker cyst sits at the posteromedial end of a sagittal one.
    lo, hi = int(SLICE_BAND[0] * (n - 1)), int(SLICE_BAND[1] * (n - 1))
    idx = np.unique(np.linspace(lo, hi, n_slice).astype(int)) if hi > lo else np.array([n // 2])
    while len(idx) < n_slice:
        idx = np.append(idx, idx[-1])

    planes = []
    for i in idx[:n_slice]:
        try:
            ds = pydicom.dcmread(os.path.join(d, files[int(i)]), force=True)
            a = ds.pixel_array.astype(np.float32)
            sl = float(getattr(ds, "RescaleSlope", 1) or 1)
            ic = float(getattr(ds, "RescaleIntercept", 0) or 0)
            a = a * sl + ic
        except Exception:
            a = None                      # no shape is known here; see below
        planes.append(a)

    # A slice that would not decode has no shape of its own, and inventing one is how a
    # single unreadable file erases a whole series: a substitute allocated at the resize
    # target while the decoded slices are still native makes the shape check below take
    # the substitute as the authority and zero the good slices with it, leaving a black
    # slot that the presence mask still reports as acquired.
    #
    # A failure is instead filled from the nearest slice that did decode - the same
    # convention the sampler already uses when the band holds fewer distinct slices than
    # were asked for - and a series where nothing decodes is reported absent, which the
    # mask can express, rather than black, which it cannot.
    got = [k for k, p in enumerate(planes) if p is not None]
    if RULES["decode_fill"] == "zero":
        # What an imported member was fitted with: a failure becomes a zero plane at the
        # resize target, which the shape check below then propagates to the whole slot.
        # It is the behaviour the paragraph above describes and rejects, kept here only
        # because that member's weights were learned against slots blacked out this way.
        if not got:
            DECODE_FAILED.append(rec.get("SeriesInstanceUID", d))
        planes = [np.zeros((out_size, out_size), np.float32) if p is None else p
                  for p in planes]
        got = list(range(len(planes)))
    if not got:
        DECODE_FAILED.append(rec.get("SeriesInstanceUID", d))
        return None
    if len(got) < len(planes):
        DECODE_FAILED.append(rec.get("SeriesInstanceUID", d))
        for k, p in enumerate(planes):
            if p is None:
                planes[k] = planes[min(got, key=lambda j: abs(j - k))]

    # Slices of one series can still differ in matrix size - multi-echo and some
    # reformats do - and those are genuinely not stackable.
    shp = planes[0].shape
    planes = [p if p.shape == shp else np.zeros(shp, np.float32) for p in planes]
    vol = np.stack(planes)

    # constant physical extent, then resize: PixelSpacing varies 3.4x across the corpus
    if px and np.isfinite(px) and px > 0:
        want = int(round(CROP_MM / px))
        h, w = shp
        if 16 < want < min(h, w):
            cy, cx = h // 2, w // 2
            half = want // 2
            vol = vol[:, max(0, cy - half):cy + half, max(0, cx - half):cx + half]

    lo_v, hi_v = np.percentile(vol, [1, 99])
    vol = np.clip((vol - lo_v) / max(hi_v - lo_v, 1e-6), 0, 1)

    t = torch.from_numpy(np.ascontiguousarray(vol)).unsqueeze(0)
    t = F.interpolate(t, size=(out_size, out_size), mode="bilinear", align_corners=False)
    # uint8, not float32. These buffers queue up between the reader threads and the
    # encoder, and at this size a float32 slot-series is several megabytes. Intensity is
    # already normalised into [0, 1] here, so eight bits cost nothing that a bilinear
    # resize has not already cost, and the queue is a quarter the size.
    return (t.squeeze(0) * 255).round().clamp(0, 255).to(torch.uint8)


## 6. Laterality normalisation

In [ ]:
def normalise_laterality(img, plane, lat):
    """Map every knee onto a left-knee convention.

    Coronal and axial views mirror under a horizontal flip. Sagittal stacks are not
    mirror images of each other - the slice order runs medial-to-lateral in opposite
    directions - so the channel order is reversed instead.
    """
    if lat != "R":
        return img
    if plane in ("Coronal", "Axial"):
        return torch.flip(img, dims=[-1])
    return torch.flip(img, dims=[0])


## 7. Decode each selected series once into the study cache

In [ ]:
# Where the geometric slice order may be remembered between runs. Unset on the platform,
# because each run gets a fresh machine and there is nothing to remember; set off it,
# where the same corpus is cached again at every resolution and slice count and the order
# is a function of neither. It is opt-in so that the scored run's behaviour is decided by
# the code rather than by whether a file happens to be lying about.
ORDER_CACHE = os.environ.get("RSNA_ORDER_CACHE") or None


def build_cache(slot_map, plane_map, lat_map, tag):
    """Decode every (study, slot) once into an in-memory uint8 array.

    Fine-tuning revisits the same pixels every epoch. Reading them from the mount each
    time would make the epoch count a function of I/O rather than of learning, so they
    are decoded once and held as bytes: intensity has already been normalised into
    [0, 1], and eight bits cost nothing a bilinear resize has not already cost.

    CACHE_SLICES positions are kept per slot, which the training loop reads as N_GROUP
    groups of GROUP consecutive channels.
    """
    studies = sorted(slot_map)
    sidx = {s: i for i, s in enumerate(studies)}
    cache = np.zeros((len(studies), N_SLOT, CACHE_SLICES, IMG, IMG), np.uint8)
    mask = np.zeros((len(studies), N_SLOT), np.float32)
    log(f"{tag}: cache {cache.shape} = {cache.nbytes / 1024 ** 3:.1f} GB")

    jobs = [(st, k, plane, slot_map[st][name])
            for st in studies
            for k, (name, plane, _, _) in enumerate(SLOTS)
            if name in slot_map[st]]
    n_job = len(jobs)

    # Ordering first, and as its own pass. It reads one header per slice of every chosen
    # series - far more file opens than the decode that follows - and on a network mount
    # that is latency, not work, so it gets its own wider pool.
    t_ord = time.time()
    n_slice_total = sum(len(j[3]["files"]) for j in jobs)
    log(f"{tag}: ordering {len(jobs)} slot-series ({n_slice_total} slice headers)")
    ok = done = 0
    CHUNK_O = 1024

    # A remembered order, when one is offered. The projection depends on the DICOM
    # geometry alone, so it is the same at every resolution and every slice count, and
    # it costs one header read per slice - the largest single cost in this pass. An entry
    # is validated by the number of files present, so a tree that has changed under it is
    # recomputed rather than trusted: order is derived data, and a stale entry would be
    # invisible in the way that matters most.
    seen = {}
    if ORDER_CACHE and Path(ORDER_CACHE).is_file():
        try:
            import json as _json
            seen = _json.loads(Path(ORDER_CACHE).read_text())
        except (OSError, ValueError):
            seen = {}
        hit = 0
        for _, _, _, rec in jobs:
            e = seen.get(rec["SeriesInstanceUID"])
            if e and len(e["files"]) == len(rec["files"]):
                rec["ordered"] = e["files"]
                ok += int(e["good"])
                hit += 1
        jobs = [j for j in jobs if "ordered" not in j[3]]
        log(f"{tag}: {hit} slot-series ordered from {ORDER_CACHE}, {len(jobs)} to read")

    with ThreadPoolExecutor(max_workers=ORDER_THREADS) as pool:
        for c0 in range(0, len(jobs), CHUNK_O):
            block = jobs[c0:c0 + CHUNK_O]
            for (_, _, _, rec), (files, good) in zip(
                    block, pool.map(lambda j: order_slices(j[3]), block)):
                rec["ordered"] = files
                ok += int(good)
                done += 1
                if ORDER_CACHE:
                    seen[rec["SeriesInstanceUID"]] = {"files": files, "good": bool(good)}
            # The ceiling is whichever comes first: the pass's own budget, or the share
            # of what is left of the run that it may take. The second is what makes the
            # first safe to set generously - a mount slow enough to matter cannot spend
            # the training time, because the budget shrinks as the run does.
            budget = min(ORDER_BUDGET_S, max(60.0, (TIME_BUDGET - (time.time() - T0)) * 0.35))
            if time.time() - t_ord > budget:
                log(f"{tag}: ordering budget spent at {done}/{len(jobs)}; "
                    f"the rest keep file order")
                break
    if ORDER_CACHE and done:
        import json as _json
        _t = Path(ORDER_CACHE).with_suffix(".tmp")
        _t.write_text(_json.dumps(seen))
        _t.replace(Path(ORDER_CACHE))
    log(f"{tag}: ordered {ok}/{n_job} by geometry "
        f"({n_job - ok} kept arbitrary) in {time.time() - t_ord:.0f}s")

    jobs = [(st, k, plane, slot_map[st][name])
            for st in studies
            for k, (name, plane, _, _) in enumerate(SLOTS)
            if name in slot_map[st]]
    log(f"{tag}: decoding {len(jobs)} slot-series")
    n_failed_before = len(DECODE_FAILED)

    CHUNK = 512
    done = 0
    with ThreadPoolExecutor(max_workers=PIX_THREADS) as pool:
        for c0 in range(0, len(jobs), CHUNK):
            block = jobs[c0:c0 + CHUNK]
            for (st, k, plane, _), img in zip(
                    block, pool.map(lambda j: read_slot(j[3], CACHE_SLICES, IMG), block)):
                done += 1
                if img is None:
                    continue
                cache[sidx[st], k] = normalise_laterality(img, plane,
                                                          lat_map.get(st)).numpy()
                mask[sidx[st], k] = 1.0
            if done % 4096 < CHUNK:
                log(f"  {tag} {done}/{len(jobs)}")
            if time.time() - T0 > TIME_BUDGET:
                log(f"  {tag}: time budget reached during decode")
                break
    n_failed = len(DECODE_FAILED) - n_failed_before
    log(f"{tag}: {int(mask.sum())}/{len(jobs)} slots filled"
        + (f"; {n_failed} series had a slice that would not decode" if n_failed else ""))
    gc.collect()
    return studies, cache, mask


## 8. Checkpoint-compatible DINOv2 multi-slot model

In [ ]:
class SlotHead(nn.Module):
    """Per-diagnosis attention over the slot embeddings of one study.

    Each finding is read on particular sequences - cruciates sagittally, collateral
    ligaments and the meniscal body coronally, patellar cartilage axially - so pooling
    the slots identically would dilute the one that carries the evidence with the rest.

    The aggregation is deliberately this simple. With a study-level label there is no
    signal telling the model which part of a study matters, so extra attention
    parameters below the slot level would have nothing to learn from and would spend
    their capacity fitting noise.
    """

    def __init__(self, dim, n_slot, n_out, hidden=256, p=0.2, prior=False):
        super().__init__()
        self.proj = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, hidden), nn.GELU())
        self.slot_emb = nn.Parameter(torch.randn(n_slot, hidden) * 0.02)
        self.query = nn.Parameter(torch.randn(n_out, hidden) * 0.02)
        self.drop = nn.Dropout(p)
        self.out = nn.Linear(hidden, n_out)
        self.hidden = hidden
        # An imported member carries a fixed per-(diagnosis, slot) tilt on the attention
        # logits, set from the anatomy table below rather than learned. It is a buffer, so
        # it travels in the state dict and must exist for that member to load; exp(0.55)
        # gives a preferred slot about 1.73x the weight of an unpreferred one, which
        # biases the softmax without ever excluding a slot.
        p_ = torch.zeros(n_out, n_slot)
        if prior and n_slot == len(SLOTS) and n_out == len(TARGETS):
            for t, slots in SLOT_PRIOR_TABLE.items():
                if t in TARGETS:
                    p_[TARGETS.index(t), list(slots)] = SLOT_PRIOR_STRENGTH
        self.prior = prior
        if prior:
            self.register_buffer("slot_prior", p_)

    def forward(self, x, mask):
        h = self.proj(x) + self.slot_emb
        att = torch.einsum("bsh,oh->bos", h, self.query) / self.hidden ** 0.5
        if self.prior:
            att = att + self.slot_prior.unsqueeze(0)
        att = att.masked_fill(mask.unsqueeze(1) < 0.5, -1e4).softmax(-1)
        ctx = self.drop(torch.einsum("bos,bsh->boh", att, h))
        return (ctx * self.out.weight.unsqueeze(0)).sum(-1) + self.out.bias


In [ ]:
class Model(nn.Module):
    """Encoder plus head, trained end to end.

    A study arrives as a bag of slot images. The bag is flattened for the encoder and
    folded back before the head, so the encoder never sees the study structure and the
    head never sees pixels.
    """

    def __init__(self, backbone, dim, pool="cls_mean", prior=False):
        super().__init__()
        self.backbone = backbone
        self.pool = pool
        self.head = SlotHead(dim * POOL_PARTS[pool], N_SLOT, len(TARGETS), prior=prior)
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def forward(self, imgs, mask, img_size=None):
        B, S = imgs.shape[:2]
        x = imgs.reshape(B * S, *imgs.shape[2:]).float().div_(255.0)
        if img_size is not None and img_size != x.shape[-1]:
            # The cache is held at the highest resolution any configuration needs; the
            # rest downsample from it, so every configuration sees the same pixels
            # through a different sampling grid rather than a different crop.
            x = F.interpolate(x, size=(img_size, img_size), mode="bilinear",
                              align_corners=False)
        x = (x - self.mean) / self.std
        out = self.backbone(pixel_values=x).last_hidden_state
        patch = out[:, 1:]
        parts = [out[:, 0], patch.mean(1)]
        if self.pool == "cls_mean_focal":
            # The upper tail of each channel over the patch grid, taken per channel
            # rather than by selecting whole patches: a finding occupies a small part of
            # the field, so a plain mean over 256 patches dilutes it by two orders of
            # magnitude, and this keeps the top eighth of each channel's responses.
            k = max(1, patch.shape[1] // 8)
            parts.append(patch.topk(k, dim=1).values.mean(1))
        feat = torch.cat(parts, dim=1).reshape(B, S, -1)
        return self.head(feat, mask)


In [ ]:
def build_model(unfreeze_last, source=None, variant="small", pool="cls_mean",
                prior=False):
    """Load the encoder and open the last `unfreeze_last` blocks for training.

    The early blocks of a self-supervised transformer are generic edge and texture
    filters; the late blocks carry semantics. Opening only the late ones is the cautious
    choice - there may not be enough supervision here to improve the early ones and there
    is certainly enough to damage them - but how far the line should sit is a question
    the corpus has to answer rather than the intuition.

    `source` names where the weights come from. Left unset it is the attached model
    directory, which is the only thing available here. It is a parameter so that a run
    off the platform builds the same object from the same code rather than from a second
    definition that has to be kept in step by hand.
    """
    from transformers import AutoModel
    p = source if source is not None else find_dinov2(variant)
    if p is None:
        raise FileNotFoundError("DINOv2 weights not attached")
    bb = AutoModel.from_pretrained(str(p))
    n_layer = len(bb.encoder.layer)
    for prm in bb.parameters():
        prm.requires_grad = False
    for blk in bb.encoder.layer[max(0, n_layer - unfreeze_last):]:
        for prm in blk.parameters():
            prm.requires_grad = True
    for prm in bb.layernorm.parameters():
        prm.requires_grad = True
    dim = bb.config.hidden_size
    trainable = sum(p.numel() for p in bb.parameters() if p.requires_grad)
    log(f"backbone: {n_layer} blocks, last {unfreeze_last} trainable "
        f"({trainable / 1e6:.1f}M params), feature dim {dim * POOL_PARTS[pool]}")
    return Model(bb, dim, pool=pool, prior=prior)


## 8.5 Gap analysis from the supplied notebooks

The important differences are architectural and validation-related, not cosmetic:

| area | current 0.891 enhanced notebook | stronger lesson from supplied notebooks | action here |
|---|---|---|---|
| window aggregation | mean probability + small window-rank vote | focal targets use max/top-2 window evidence | **adopted** |
| member aggregation | default scalar holdout weighting because target AUC metadata is absent | equal rank voting is safer unless target-level OOF evidence exists | **adopted** |
| backbone diversity | 20 closely related DINOv2 members | independent EfficientNet-B3 family reaches a different error regime | **optional audited blend** |
| label quality | fixed upstream labels/checkpoints | stronger systems improve report-derived supervision before image training | requires retraining |
| validation | inference heuristic not directly OOF-selected per target | grouped folds + strict expert-label OOF / nested selection | audit gate |
| spatial resolution | already 336 px / 130 mm crop | high resolution matters for meniscal pathology | already strong |
| test-time text | unavailable | reports should supervise training, not be required at test time | unchanged |

The central conclusion is that another arbitrary ensemble coefficient is unlikely to
close a large leaderboard gap. The next gains should come from **local evidence pooling,
independent model families, and better supervision**, with OOF evidence deciding what is
promoted.


## 9. Final evidence-first ensemble

### Gap fixed from the 0.891 notebook

The previous `hybrid` path had no per-target AUC metadata in the attached 20 checkpoints,
so all members fell back to a scalar holdout score plus a small TTA-window rank vote.
That is not the same as showing that a specific diagnosis benefits from a specific
aggregation rule.

This version instead uses **target-level window pooling before rank ensembling**:

| target | pooling |
|---|---|
| Fracture | max window |
| Contusion | max window |
| Medial Meniscus | max window |
| Lateral Meniscus | max window |
| Baker's cyst | max window |
| ACL | mean of top 2 windows |
| MCL | mean of top 2 windows |
| all others | mean of all windows |

The rule is especially plausible for focal pathology: averaging ten windows can dilute a
tear/fracture that is visible in only one or two locations.

### Cross-family diversity

If a complete audited EfficientNet-B3 five-fold package is attached, the notebook also
creates:

- `submission_dino_b3_10.csv`: 90% DINO frontier rank + 10% B3 rank
- `submission_dino_b3_target_candidate.csv`: target-wise experimental blend

Only the **global 10%** blend is eligible to become `submission.csv` automatically, and
only after the B3 audit shows nested OOF improvement over its DINO reference. Otherwise the
DINO frontier stays primary.


In [ ]:

# -----------------------------------------------------------------------------
# Final evidence-first inference.
# Primary DINO recipe:
#   * overlapping windows
#   * target-specific max/top2 pooling for focal diagnoses
#   * equal per-member percentile-rank mean
# Optional:
#   * independently trained EfficientNet-B3 five-fold family
#   * 10% global rank blend, promoted only through an audit gate
# -----------------------------------------------------------------------------

import math
import shutil
import subprocess
import sys

FINGERPRINT_TOL = 2e-3
TTA_OVERLAP = True
TTA_POOL = "prob"


def fingerprint(model, dev, img_size, n_slot=None, group=None, seed=None):
    n_slot = N_SLOT if n_slot is None else n_slot
    group = GROUP if group is None else group
    seed = SEED if seed is None else seed
    g = torch.Generator().manual_seed(seed)
    imgs = torch.randint(
        0, 256, (2, n_slot, group, img_size, img_size),
        generator=g, dtype=torch.uint8
    ).to(dev)
    mask = torch.ones(2, n_slot, device=dev)
    mask[1, -1] = 0.0
    was_training = model.training
    model.eval()
    with torch.no_grad():
        out = model(imgs, mask, img_size).float().cpu().numpy()
    if was_training:
        model.train()
    return out


class WeightsError(RuntimeError):
    pass


def check_fingerprint(model, dev, img_size, expected, tol=FINGERPRINT_TOL, tag=""):
    got = fingerprint(model, dev, img_size)
    exp = np.asarray(expected, np.float32)
    if got.shape != exp.shape:
        raise WeightsError(f"{tag}fingerprint shape {got.shape} != stored {exp.shape}")
    d = float(np.abs(got - exp).max())
    if d > tol:
        raise WeightsError(
            f"{tag}fingerprint differs by {d:.4g} > {tol:g}; "
            "architecture/preprocessing contract moved"
        )
    log(f"{tag}fingerprint matches within {d:.2g}")
    return d


def _valid_package(root, name="manifest.json"):
    root = Path(root)
    p = root / name
    if not p.is_file():
        return False
    try:
        man = json.loads(p.read_text())
    except Exception:
        return False
    members = man.get("members")
    if not isinstance(members, list) or not members:
        return False
    missing = [
        m.get("file") for m in members
        if not (root / str(m.get("file"))).is_file()
    ]
    if missing:
        raise WeightsError(
            f"weights package {root} is incomplete; first missing: {missing[0]}"
        )
    return True


def find_weights(name="manifest.json"):
    checked = []
    if PREFERRED_WEIGHTS:
        checked.append(str(PREFERRED_WEIGHTS))
        if _valid_package(PREFERRED_WEIGHTS, name):
            log(f"weights package found: {PREFERRED_WEIGHTS}")
            return PREFERRED_WEIGHTS

    base = Path("/kaggle/input")
    if base.is_dir():
        for root, dirs, files in os.walk(base):
            dirs[:] = [d for d in dirs if d not in ("train_series", "test_series")]
            if name not in files:
                continue
            root = Path(root)
            checked.append(str(root))
            if _valid_package(root, name):
                log(f"weights package found: {root}")
                return root

    raise FileNotFoundError(
        "No compatible manifest checkpoint package found. "
        f"Checked: {checked[:20]}"
    )


def window_starts(n_slice, group, overlap=None):
    overlap = TTA_OVERLAP if overlap is None else overlap
    if overlap and n_slice >= group:
        return list(range(n_slice - group + 1))
    return [g * group for g in range(max(n_slice // group, 1))]


def _apply_frontier_pool(probs):
    """Pool [window,batch,target] probabilities using diagnosis-specific rules."""
    v = probs.mean(dim=0)
    target_idx = {t: j for j, t in enumerate(TARGETS)}
    for target, mode in FRONTIER_TARGET_POOL.items():
        j = target_idx[target]
        x = probs[:, :, j]
        if mode == "max":
            v[:, j] = x.max(dim=0).values
        elif mode.startswith("top"):
            k = min(int(mode[3:]), x.shape[0])
            v[:, j] = x.topk(k, dim=0).values.mean(dim=0)
        elif mode == "mean":
            v[:, j] = x.mean(dim=0)
        else:
            raise ValueError(f"unknown frontier pooling mode {mode!r}")
    return v


@torch.no_grad()
def predict_member_frontier(model, cache, mask, idx, dev, img_size, group=None, starts=None):
    """Return both old mean-window predictions and target-pooled frontier predictions."""
    group = GROUP if group is None else group
    starts = window_starts(cache.shape[2], group) if starts is None else list(starts)
    if not starts:
        raise ValueError("no TTA windows")

    model.eval()
    mean_pred = np.empty((len(idx), len(TARGETS)), np.float32)
    frontier_pred = np.empty_like(mean_pred)

    for b0 in range(0, len(idx), EVAL_BATCH):
        sel = idx[b0:b0 + EVAL_BATCH]
        b1 = b0 + len(sel)
        m = torch.from_numpy(mask[sel]).to(dev)
        win = []

        for st in starts:
            rows = torch.from_numpy(
                np.ascontiguousarray(cache[sel, :, st:st + group])
            ).to(dev)
            with torch.autocast("cuda", enabled=dev.type == "cuda"):
                z = model(rows, m, img_size).float()
            win.append(torch.sigmoid(z))

        probs = torch.stack(win, dim=0)
        mean_pred[b0:b1] = probs.mean(dim=0).cpu().numpy()
        frontier_pred[b0:b1] = _apply_frontier_pool(probs).cpu().numpy()

    return mean_pred, frontier_pred


def _rank_matrix(x):
    return (
        pd.DataFrame(np.asarray(x))
        .rank(method="average", pct=True)
        .to_numpy(np.float64)
    )


def adopt_config_globals(cfg):
    global IMG, CACHE_IMG, GROUP, CACHE_SLICES, N_GROUP, CROP_MM, SLICE_BAND, RULES
    CACHE_IMG = IMG = int(cfg["img"])
    GROUP = int(cfg["group"])
    CACHE_SLICES = int(cfg["slices"])
    N_GROUP = max(CACHE_SLICES // GROUP, 1)
    CROP_MM = float(cfg["crop_mm"])
    SLICE_BAND = tuple(float(x) for x in cfg["band"])
    rules = cfg.get("rules") or RULES_NATIVE
    unknown = {
        k: v for k, v in rules.items()
        if k not in RULES_NATIVE or v not in (RULES_NATIVE[k], RULES_LEGACY[k])
    }
    if unknown:
        raise WeightsError(f"unsupported pixel rules in manifest: {unknown}")
    RULES = {**RULES_NATIVE, **rules}
    if [s[0] for s in SLOTS] != list(cfg["slots"]):
        raise WeightsError(
            f"manifest slots {cfg['slots']} != notebook slots {[s[0] for s in SLOTS]}"
        )


def write_submission(pred, studies, test_df, path):
    ranked = _rank_matrix(pred)
    sub = pd.DataFrame(ranked, columns=TARGETS)
    sub.insert(0, "StudyInstanceUID", studies)
    sub = test_df[["StudyInstanceUID"]].merge(
        sub, on="StudyInstanceUID", how="left"
    )
    sub[TARGETS] = sub[TARGETS].fillna(0.5)
    sub.to_csv(path, index=False)
    return sub


def write_benchmark_submission():
    t = pd.read_csv(ROOT / "test.csv")
    for c in TARGETS:
        t[c] = 0.5
    t.to_csv("submission.csv", index=False)


def _validate_submission_frame(frame, test_df, tag):
    expected = ["StudyInstanceUID", *TARGETS]
    if frame.columns.tolist() != expected:
        raise RuntimeError(f"{tag}: schema mismatch")
    uid = test_df["StudyInstanceUID"].astype(str).tolist()
    got = frame["StudyInstanceUID"].astype(str).tolist()
    if got != uid:
        # Reorder only if the UID set is exact.
        if set(got) != set(uid) or len(got) != len(uid):
            raise RuntimeError(f"{tag}: hidden UID set mismatch")
        frame = (
            test_df[["StudyInstanceUID"]]
            .astype({"StudyInstanceUID": str})
            .merge(frame.astype({"StudyInstanceUID": str}),
                   on="StudyInstanceUID", how="left")
        )
    if frame["StudyInstanceUID"].duplicated().any():
        raise RuntimeError(f"{tag}: duplicate StudyInstanceUID")
    arr = frame[TARGETS].to_numpy(np.float64)
    if not np.isfinite(arr).all():
        raise RuntimeError(f"{tag}: non-finite prediction")
    return frame


def _find_b3_package():
    explicit = os.environ.get("KNEE_B3_DIR", "").strip()
    candidates = []
    if explicit:
        candidates.append(Path(explicit))
    candidates.append(Path("/kaggle/input/rsna-knee-b3-v47-folds-0-3"))

    base = Path("/kaggle/input")
    if base.is_dir():
        for p in base.iterdir():
            if p.is_dir() and "b3" in p.name.lower():
                candidates.append(p)

    seen = set()
    for root in candidates:
        root = root.resolve() if root.exists() else root
        if str(root) in seen:
            continue
        seen.add(str(root))
        infer_py = root / "source/efficientnet_b3_public_repro_v1_infer.py"
        module_py = root / "source/efficientnet_b3_public_repro_v4_t4.py"
        folds = [root / f"fold{i}/fold{i}_final.pt" for i in range(5)]
        if infer_py.is_file() and module_py.is_file() and all(p.is_file() for p in folds):
            return root
    return None


def _b3_audit_supports_blend(root):
    audit_path = root / "audit/audit.json"
    if not audit_path.is_file():
        return False, "audit/audit.json absent"
    try:
        audit = json.loads(audit_path.read_text())
        nested = float(audit["selection"]["global_nested_macro_auc"])
        base = float(audit["arms"]["exact_public_macro_auc"])
        if nested > base:
            return True, f"nested OOF {nested:.5f} > DINO reference {base:.5f}"
        return False, f"nested OOF {nested:.5f} <= DINO reference {base:.5f}"
    except Exception as exc:
        return False, f"audit parse failed: {type(exc).__name__}: {exc}"


def _run_b3_candidate(dino_sub, test_df):
    """Run optional five-fold B3 inference and create global + target-wise rank blends."""
    root = _find_b3_package()
    if root is None:
        log("B3 package not found; keeping DINO frontier primary")
        return None, None, False

    supports, audit_msg = _b3_audit_supports_blend(root)
    log(f"B3 package: {root}; audit: {audit_msg}")

    if not torch.cuda.is_available():
        log("B3 candidate skipped: CUDA unavailable")
        return None, None, False

    left = TIME_BUDGET - (time.time() - T0)
    if left < 15 * 60:
        log(f"B3 candidate skipped: only {left/60:.1f} min remain")
        return None, None, False

    outdir = Path("/kaggle/working/rsna_b3_final_inference")
    outdir.mkdir(parents=True, exist_ok=True)
    infer_py = root / "source/efficientnet_b3_public_repro_v1_infer.py"
    module_py = root / "source/efficientnet_b3_public_repro_v4_t4.py"
    folds = [root / f"fold{i}/fold{i}_final.pt" for i in range(5)]

    budget_hours = min(1.75, max(0.25, 0.90 * left / 3600.0))
    cmd = [
        sys.executable, str(infer_py),
        "--module", str(module_py),
        "--test-csv", str(ROOT / "test.csv"),
        "--series-csv", str(ROOT / "test_series.csv"),
        "--image-root", str(ROOT / "test_series"),
        "--checkpoints", *map(str, folds),
        "--output-dir", str(outdir),
        "--budget-hours", f"{budget_hours:.6f}",
        "--checkpoint-every", "10",
    ]
    log(f"running B3 candidate with {budget_hours:.2f}h adaptive budget")
    res = subprocess.run(
        cmd,
        timeout=max(60.0, min(left * 0.96, budget_hours * 3600 + 10 * 60)),
        check=False,
    )
    if res.returncode != 0:
        log(f"B3 candidate failed with exit={res.returncode}; keeping DINO")
        return None, None, False

    b3_path = outdir / "submission.csv"
    if not b3_path.is_file():
        log("B3 candidate did not write submission.csv; keeping DINO")
        return None, None, False

    b3 = _validate_submission_frame(
        pd.read_csv(b3_path, dtype={"StudyInstanceUID": str}),
        test_df.astype({"StudyInstanceUID": str}),
        "B3"
    )
    dino = _validate_submission_frame(
        dino_sub.astype({"StudyInstanceUID": str}).copy(),
        test_df.astype({"StudyInstanceUID": str}),
        "DINO frontier"
    )

    dino_rank = dino[TARGETS].rank(method="average", pct=True)
    b3_rank = b3[TARGETS].rank(method="average", pct=True)

    global_blend = dino.copy()
    global_blend[TARGETS] = (
        (1.0 - B3_GLOBAL_ALPHA) * dino_rank
        + B3_GLOBAL_ALPHA * b3_rank
    )
    global_path = "submission_dino_b3_10.csv"
    global_blend.to_csv(global_path, index=False)

    target_blend = dino.copy()
    for target in TARGETS:
        a = float(B3_TARGET_ALPHAS[target])
        target_blend[target] = (
            (1.0 - a) * dino_rank[target] + a * b3_rank[target]
        )
    target_path = "submission_dino_b3_target_candidate.csv"
    target_blend.to_csv(target_path, index=False)

    promote = supports or ALLOW_UNAUDITED_B3
    log(
        f"B3 candidates written; auto-promote={promote} "
        f"(audit_supported={supports}, allow_unaudited={ALLOW_UNAUDITED_B3})"
    )
    return global_blend, target_blend, promote


def infer_from_package_final(path, dev):
    man = json.loads((Path(path) / "manifest.json").read_text())
    members = man["members"]
    log(f"weights package: {len(members)} DINO member(s) from {path}")

    test_df = pd.read_csv(ROOT / "test.csv")
    test_series = pd.read_csv(ROOT / "test_series.csv")
    plane_map = dict(zip(
        test_series["SeriesInstanceUID"], test_series["Anatomical_Plane"]
    ))
    hte = annotate(walk("test_series"))
    log(f"test header pass: {len(hte)} series")
    audit_official_sequence_metadata(hte, test_series)

    groups = {}
    for m in members:
        groups.setdefault(m["pixel_group"], []).append(m)

    per_member = []
    fixed_s = per_win_s = None
    group_items = list(groups.items())

    for gi, (key, gm) in enumerate(group_items, 1):
        cfg = json.loads(key)
        adopt_config_globals(cfg)
        log(
            f"decode group {gi}/{len(group_items)}: "
            f"{cfg['img']}px x {cfg['slices']} slices, "
            f"crop {cfg['crop_mm']} mm -> {len(gm)} member(s)"
        )

        st_te, Cte, Mte = build_cache(
            pick_slots(hte, plane_map),
            plane_map,
            lat_of(hte, "test "),
            f"test g{gi}"
        )
        idx = np.arange(len(st_te))
        starts = window_starts(Cte.shape[2], GROUP)
        order = sorted(gm, key=lambda m: -(m.get("holdout") or 0))
        left_after = sum(len(g) for _, g in group_items[gi:])

        for k, m in enumerate(order):
            left = TIME_BUDGET - (time.time() - T0)
            remaining = (len(order) - k) + left_after
            use_starts = list(starts)

            if fixed_s is not None and per_win_s is not None:
                afford = max(left * 0.90, 0.0)
                need = fixed_s + len(use_starts) * per_win_s
                if need * remaining > afford:
                    room = afford / max(remaining, 1)
                    n_win = int((room - fixed_s) / per_win_s) if per_win_s > 0 else 0
                    n_win = max(1, min(len(use_starts), n_win))
                    if fixed_s + per_win_s > afford:
                        log(
                            f"{left/60:.0f} min left: stopping; "
                            "one more DINO member does not fit"
                        )
                        break
                    if n_win < len(use_starts):
                        mid = (len(use_starts) - n_win) // 2
                        use_starts = use_starts[mid:mid + n_win]
                        log(
                            f"{left/60:.0f} min left: "
                            f"using {n_win}/{len(starts)} windows"
                        )

            t0 = time.time()
            ck = torch.load(
                Path(path) / m["file"], map_location="cpu", weights_only=False
            )
            model = build_model(
                int(m["config"]["unfreeze_last"]),
                variant=m["config"]["variant"],
                pool=m["config"].get("pool", "cls_mean"),
                prior=bool(m["config"].get("prior", False)),
            ).to(dev)
            model.load_state_dict(ck["model"])
            check_fingerprint(
                model, dev, IMG, ck["fingerprint"], tag=f"{m['id']}: "
            )
            t_ready = time.time()

            p_mean, p_frontier = predict_member_frontier(
                model, Cte, Mte, idx, dev, IMG, starts=use_starts
            )
            per_member.append({
                "id": m["id"],
                "fold": m.get("fold"),
                "ids": st_te,
                "mean": p_mean,
                "frontier": p_frontier,
                "holdout": m.get("holdout"),
                "n_windows": len(use_starts),
            })

            fixed_s = t_ready - t0
            per_win_s = (time.time() - t_ready) / max(len(use_starts), 1)
            log(
                f"  {m['id']} fold {m.get('fold')}: "
                f"{len(idx)} studies, {len(use_starts)} windows, "
                f"{time.time()-t0:.0f}s"
            )

            del model, ck
            gc.collect()
            if dev.type == "cuda":
                torch.cuda.empty_cache()

        del Cte, Mte
        gc.collect()

    if not per_member:
        raise WeightsError("no DINO member produced predictions")

    all_ids = sorted({s for m in per_member for s in m["ids"]})
    pos = {s: i for i, s in enumerate(all_ids)}
    M, N, T = len(per_member), len(all_ids), len(TARGETS)
    mean_rank = np.full((M, N, T), np.nan, np.float64)
    frontier_rank = np.full((M, N, T), np.nan, np.float64)

    for mi, m in enumerate(per_member):
        rows = [pos[s] for s in m["ids"]]
        mean_rank[mi, rows] = _rank_matrix(m["mean"])
        frontier_rank[mi, rows] = _rank_matrix(m["frontier"])

    if np.isnan(mean_rank).any() or np.isnan(frontier_rank).any():
        raise WeightsError("some DINO members did not cover all hidden test studies")

    # Baseline reproduces the old mean-window family; frontier is the evidence-backed
    # diagnosis-specific pooling family.
    baseline = mean_rank.mean(axis=0)
    frontier = frontier_rank.mean(axis=0)

    baseline_sub = write_submission(
        baseline, all_ids, test_df, "submission_dino_mean_baseline.csv"
    )
    frontier_sub = write_submission(
        frontier, all_ids, test_df, "submission_dino_frontier.csv"
    )

    # Establish the safe primary before optional work.
    frontier_sub.to_csv("submission.csv", index=False)

    diag = pd.DataFrame({
        "member": [m["id"] for m in per_member],
        "fold": [m.get("fold") for m in per_member],
        "holdout": [m.get("holdout") for m in per_member],
        "n_windows": [m.get("n_windows") for m in per_member],
    })
    display(diag.sort_values(["fold", "holdout"], ascending=[True, False]))

    log(
        f"DINO frontier written from {len(per_member)} member(s); "
        f"target pooling={FRONTIER_TARGET_POOL}"
    )

    if FINAL_MODE == "dino_frontier":
        log("RSNA_FINAL_MODE=dino_frontier; skipping B3")
        return frontier_sub

    b3_global, b3_target, promote = _run_b3_candidate(frontier_sub, test_df)

    if FINAL_MODE == "dino_b3_10":
        if b3_global is None:
            raise RuntimeError(
                "RSNA_FINAL_MODE=dino_b3_10 requested but B3 candidate is unavailable"
            )
        b3_target.to_csv("submission.csv", index=False)
        log("forced primary: DINO + 10% B3 rank blend")
        return b3_target

    # auto
    if b3_global is not None and promote:
        b3_target.to_csv("submission.csv", index=False)
        log("AUTO primary promoted: DINO frontier + TARGET-SPECIFIC B3 rank")
        return b3_target

    log("AUTO primary remains DINO frontier")
    return frontier_sub


def require_cuda():
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA GPU is required for this final inference notebook")
    return torch.device("cuda")


def main():
    # A valid safety artifact exists from the start.
    write_benchmark_submission()
    pkg = find_weights()
    dino = find_dinov2("small")
    if dino is None:
        raise FileNotFoundError(
            "DINOv2 base model not found. Attach the local/offline DINOv2 model "
            "dataset or set KNEE_DINOV2_DIR."
        )
    log(f"using DINOv2 base: {dino}")
    return infer_from_package_final(pkg, require_cuda())


## 10. Run inference and write submissions

In [ ]:
try:
    submission = main()
    display(submission.head())
    log("done")
except Exception:
    traceback.print_exc()
    raise


### Final outputs

The notebook always creates:

- `/kaggle/working/submission_dino_mean_baseline.csv`
- `/kaggle/working/submission_dino_frontier.csv`
- `/kaggle/working/submission.csv`

If the B3 package is available and inference succeeds, it also creates:

- `/kaggle/working/submission_dino_b3_10.csv`
- `/kaggle/working/submission_dino_b3_target_candidate.csv`

`submission.csv` selection:

- `dino_frontier` mode: diagnosis-specific DINO pooling + equal member rank mean.
- `dino_b3_10` mode: forces 90/10 DINO/B3 rank blend.
- `auto` mode (default): DINO frontier first; promotes the 10% B3 blend only when its
  attached audit supports nested OOF improvement, unless `ALLOW_UNAUDITED_B3=1`.

The target-wise B3 file is intentionally a **candidate**, not the automatic primary,
because its per-target weights were selected from a small expert-labelled set.

### 0.95 target

This notebook removes clear inference-side weaknesses and adds validated model-family
diversity when available. Reaching ~0.95, however, likely requires a stronger training
stage: better per-target weak-label fusion, fully cross-fitted expert-label validation,
and multiple independently trained backbones rather than additional inference heuristics
on the same 20 DINO checkpoints.
